In [1]:
from doc_to_graphrag.ingestion import DocumentLoader

### OCR modes

The document loader supports three **OCR modes**:

- **`tesseract`**: Pytesseract only; no Mistral API calls.
- **`mistral`**: Always use Mistral OCR (PDF: single upload; images: base64). Supports **batch** via `load_batch()`.
- **`hybrid`**: Tesseract first; if confidence is below **ocr_min_confidence**, fallback to Mistral OCR 3.0.

**Batch loading** is only available when `ocr_mode="mistral"` (uses Mistral Batch Inference API). For `tesseract` or `hybrid`, call `load()` in a loop for multiple files.

In [2]:
loader = DocumentLoader(ocr_mode="mistral", ocr_min_confidence=0.95)

### below is a test on very illegible handwritten image (using Mistral OCR 3.0)

In [3]:
# force_ai=True forces Mistral for this call (handwritten image)
result = loader.load("test-documents/funky_image.png", force_ai=True)

text = result["text"]
print(text)

Tuesday 8:30 pm.

Just had dinner. Did not get home until nearly 8 pm.
as I am now very busy at the office. Westcott came today
and is trying to raise money at last minute. I have
to hand over balance of work to the liquidators
&amp; also finish off books before shipping them to N. York
tomorrow. Glad to say it rained heavily the whole
day yesterday, which kept things quiet politically,
but of course, it was rotten getting to office back.
Went to bed at 9-20 pm. I am not going out
tonight. Still martial law, but things look better
today as the trains are running &amp; the P.O. is open &amp;
I can post this tomorrow. Will be out all day
tomorrow as I have invited 6 Chinese &amp; Mr Westcott to
tiffin. Will go to Eddie's Cafe on Broadway as I
believe it is good &amp; has music. At 6 pm. I am invited
to a Chinese dinner which M. S. H. is giving at his
home for me. I bought some socks to-day &amp; studs
for shirt. Just thought on - I gave your empty ear-rings
to Armenian shop to get Ural s

### below is a test on a rate confirmation for load (using Mistral OCR 3.0)

In [4]:
# force_ocr=True uses OCR (not pdfplumber); force_ai=True uses Mistral for whole PDF
result_2 = loader.load("test-documents/rate_confirmation_for_load.pdf", force_ocr=True, force_ai=True)

### JSON Schema if Mistral was used
```javascript
{
    "text": str,
    "mistral_ocr_response": {
        "pages": [
            {
                "index": int, # The index of the corresponding page
                "markdown": str, # The main output and raw markdown content
                "images": list, # Image information when images are extracted
                "tables": list, # Table information when using `table_format=html` or `table_format=markdown`
                "hyperlinks": list, # Hyperlinks detected
                "header": str|null, # Header content when using `extract_header=True`
                "footer": str|null, # Footer content when using `extract_footer=True`
                "dimensions": dict # The dimensions of the page
            }
        ],
    "model": str, # The model used for the OCR
    "document_annotation": dict|null, # Document annotation information when used, visit the Annotations documentation for more information
    "usage_info": dict # Usage information
    },
    "metadata": {
        "file_path": str,
        "file_type": "image",
        "page_count": int,
        "ocr_used": True,
        "ocr_confidence": 1.0,
    },
}
```

In [6]:
result_2["mistral_ocr_response"].pages

[OCRPageObject(index=0, markdown='ECHO\n\nTransportation Simplified\n\nECHODRIVE\n\nSearch, Bid, Book, Manage, Track, Get Paid.\n\nSign Up for EchoOnline Here: https://echoonline.com/\n\nDownload EchoOnline Only: Copy (Enter or Google Play only or today)\n\nECHO\n\nFor Your Use\n\n# LOAD CONFIRMATION\n\n# 24/7 DRIVER SUPPORT (855) 786-3246\n\n# Report All Issues, Delays and Additional Charges Immediately to 24/7 Driver Support Electronic Tracking Must Be Provided Throughout Transit\n\nCall the Driver Support line and ask for Load Number 57825850\n\n[tbl-0.md](tbl-0.md)\n\nPursuant to our verbal agreement of 4/19/2024 between Echo Global Logistics, hereafter referred to as ECHO, and RMZ TRANSPORT INC, MC934334/DOT2798127, hereafter referred to as CARRIER. Both parties agree that Broker\'s load number 57825850, moving on 04/23/2024 from KINGSPORT, TN to MEXICALI, BC (number of stops shown below) will move at the following rate:\n\n[tbl-1.md](tbl-1.md)\n[tbl-2.md](tbl-2.md)\n\nBY MEANS OF

### Batch loading (Mistral mode only)

`load_batch(file_paths)` is only available when `ocr_mode="mistral"`. It uses the [Mistral Batch Inference API](https://docs.mistral.ai/capabilities/batch/) for cost-effective OCR at scale. For Tesseract or Hybrid, process multiple files by calling `load()` in a loop.

In [7]:
# Example: batch load with Mistral (requires ocr_mode="mistral")
loader_mistral = DocumentLoader(ocr_mode="mistral")
results = loader_mistral.load_batch([
    "test-documents/funky_image.png",
    "test-documents/rate_confirmation_for_load.pdf",
])
for r in results:
    print(r["metadata"]["file_path"], "->", len(r["text"]), "chars")

SDKError: API error occurred: Status 402. Body: {"detail": "You cannot launch batch jobs this big with your free trial. Reduce the number of steps in your configuration or subscribe via the console."}